In [23]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

In [24]:
df = pd.read_csv('uybor.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 518 entries, 0 to 517
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Unnamed: 0.1        518 non-null    int64
 1   Unnamed: 0          518 non-null    int64
 2   Title               518 non-null    str  
 3   Square              518 non-null    str  
 4   Price               518 non-null    str  
 5   Room                518 non-null    str  
 6   Floor               518 non-null    int64
 7   Building floor      518 non-null    int64
 8   Renovation          518 non-null    str  
 9   Material            518 non-null    str  
 10  Address             518 non-null    str  
 11  Lift                518 non-null    int64
 12  Bathroom            518 non-null    int64
 13  Washing machine     518 non-null    int64
 14  TV                  518 non-null    int64
 15  Microwave oven      518 non-null    int64
 16  Playground          518 non-null    int64
 17  Furnitur

In [25]:
df.drop(['Title','Unnamed: 0.1','Unnamed: 0'],axis=1, inplace=True)

In [26]:
def price(value):
    value = value.replace(' у.е.', '').replace(' ','')
    return float(value)
df['Price'] = df['Price'].apply(price)

In [27]:
def size(value):
    value = value.replace(' м²', '').replace('м²', '').replace(' ', '')
    return float(value)

df['Square'] = df['Square'].apply(size)

In [28]:
def room(value):
    value = value.replace('+', '').strip()
    return int(value)
df['Room'] = df['Room'].apply(room)

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 518 entries, 0 to 517
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Square              518 non-null    float64
 1   Price               518 non-null    float64
 2   Room                518 non-null    int64  
 3   Floor               518 non-null    int64  
 4   Building floor      518 non-null    int64  
 5   Renovation          518 non-null    str    
 6   Material            518 non-null    str    
 7   Address             518 non-null    str    
 8   Lift                518 non-null    int64  
 9   Bathroom            518 non-null    int64  
 10  Washing machine     518 non-null    int64  
 11  TV                  518 non-null    int64  
 12  Microwave oven      518 non-null    int64  
 13  Playground          518 non-null    int64  
 14  Furniture           518 non-null    int64  
 15  Refrigerator        518 non-null    int64  
 16  Air conditioner    

In [30]:
x = df.drop('Price', axis=1)
y = df['Price']

In [31]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [32]:
numerical_features = x_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = x_train.select_dtypes(include=['object']).columns

C:\Users\User\AppData\Local\Temp\ipykernel_1388\2571878621.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = x_train.select_dtypes(include=['object']).columns


In [33]:
print(numerical_features)
print(categorical_features)

Index(['Square', 'Room', 'Floor', 'Building floor', 'Lift', 'Bathroom',
       'Washing machine', 'TV', 'Microwave oven', 'Playground', 'Furniture',
       'Refrigerator', 'Air conditioner', 'Wifi', 'Cable', 'Security',
       'Video surveillance', 'Parking space'],
      dtype='str')
Index(['Renovation', 'Material', 'Address'], dtype='str')


In [34]:
sc = StandardScaler()
oe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
num_transformer = Pipeline(
    steps=[
      ('imputer', SimpleImputer(strategy='mean')),
      ('scaler', sc)
     ]
    )

cat_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', oe)
    ]
    )

In [35]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_features),
        ('cat', cat_transformer, categorical_features)
    ]
)

In [36]:
x_train_preprocessed = preprocessor.fit_transform(x_train)
y_train_preprocessed = np.log1p(y_train)

In [37]:
x_test_preprocessed = preprocessor.transform(x_test)
y_test_preprocessed = np.log1p(y_test.values)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [38]:
x_train_tensor = torch.FloatTensor(x_train_preprocessed)
x_test_tensor = torch.FloatTensor(x_test_preprocessed)
y_train_tensor = torch.FloatTensor(y_train_preprocessed.values).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test_preprocessed).reshape(-1, 1)

In [39]:

print("\n✅ Tensors created from PREPROCESSED data!")
print(f"x_train_tensor: {x_train_tensor.shape}")
print(f"x_test_tensor: {x_test_tensor.shape}")
print(f"y_train_tensor: {y_train_tensor.shape}")
print(f"y_test_tensor: {y_test_tensor.shape}")


✅ Tensors created from PREPROCESSED data!
x_train_tensor: torch.Size([414, 441])
x_test_tensor: torch.Size([104, 441])
y_train_tensor: torch.Size([414, 1])
y_test_tensor: torch.Size([104, 1])


In [40]:
class HomePrice(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.layer1 = nn.Linear(input_size, 32)
        self.layer2 = nn.Linear(32,16)
        self.layer3 = nn.Linear(16,8)
        self.layer4 = nn.Linear(8,4)
        self.layer5 = nn.Linear(4,2)
        self.layer6 = nn.Linear(2,1)
        self.dropout = nn.Dropout(0.2)
    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.dropout(x)
        x = torch.relu(self.layer2(x))
        x = self.dropout(x)
        x = torch.relu(self.layer3(x))
        x = self.dropout(x)
        x = torch.relu(self.layer4(x))
        x = self.dropout(x)
        x = torch.relu(self.layer5(x))
        x = self.dropout(x)
        x = self.layer6(x)

        return x

In [41]:
input_size = x_train_tensor.shape[1]
model = HomePrice(input_size)

In [42]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(),lr=0.0001)

In [43]:
num_epoch = 200
batch_size = 64
train_loss = []
test_losses = []

In [44]:
for i in range(num_epoch):
    model.train()
    epoch_loss = 0
    for j in range(0, len(x_train_tensor), batch_size):
        batch_x = x_train_tensor[j:j+batch_size]
        batch_y = y_train_tensor[j:j+batch_size]
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_train_loss = epoch_loss / (len(x_train_tensor) / batch_size)

    with torch.no_grad():
        test_outputs = model(x_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_losses.append(test_loss.item())

        if (i+1) % 20 == 0:
            print(f"Epoch {i+1} / {num_epoch} Train loss: {avg_train_loss} Test loss: {test_loss.item()}")

    model.eval()
    with torch.no_grad():
        train_outputs = model(x_train_tensor)
        train_loss = criterion(train_outputs, y_train_tensor)
        train_mse = mean_squared_error(y_train_tensor.numpy(), train_outputs.numpy())
        train_rmse = np.sqrt(train_mse)

        test_outputs = model(x_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_mse = mean_squared_error(y_test_tensor.numpy(), test_outputs.numpy())
        test_rmse = np.sqrt(test_mse)

        print("training loss:", train_loss.item())
        print("training MSE:", train_mse)
        print("training RMSE:", train_rmse)
        print("test loss:", test_loss.item())
        print("test MSE:", test_mse)
        print("test RMSE:", test_rmse)

    

training loss: 111.29037475585938
training MSE: 111.2903823852539
training RMSE: 10.549425689830414
test loss: 110.84895324707031
test MSE: 110.84895324707031
test RMSE: 10.528482950884724
training loss: 111.25103759765625
training MSE: 111.25103759765625
training RMSE: 10.547560741595957
test loss: 110.80968475341797
test MSE: 110.80968475341797
test RMSE: 10.52661791618837
training loss: 111.21180725097656
training MSE: 111.21180725097656
training RMSE: 10.54570088950832
test loss: 110.77063751220703
test MSE: 110.77063751220703
test RMSE: 10.52476306204596
training loss: 111.1727066040039
training MSE: 111.17269897460938
training RMSE: 10.543846498057972
test loss: 110.73179626464844
test MSE: 110.73178100585938
test RMSE: 10.522916943787942
training loss: 111.13318634033203
training MSE: 111.1331787109375
training RMSE: 10.541972240095186
test loss: 110.69253540039062
test MSE: 110.69252014160156
test RMSE: 10.521051285000068
training loss: 111.09333801269531
training MSE: 111.0933